In [6]:
import torch
import transformers
import itertools
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = transformers.BertModel.from_pretrained('google-bert/bert-base-multilingual-cased', cache_dir="/data/user_data/jiaruil5/.cache/").to(device)
tokenizer = transformers.BertTokenizer.from_pretrained('google-bert/bert-base-multilingual-cased', cache_dir="/data/user_data/jiaruil5/.cache/")

/data/user_data/jiaruil5/miniconda3/envs/mmc/lib/python3.10/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [22]:
"嗨，我是埃琳娜，我将展示我们的工作，西班牙语中未同化借用词的检测：一个注释语料库和建模方法。"[23]

'未'

In [24]:
from nltk import word_tokenize
word_tokenize("Comme vous pouvez le voir ici, nous avons ah dans le premier exemple, nous avons le terme emprunté \"batch cooking\" qui est un emprunt de plusieurs mots.")

['Comme',
 'vous',
 'pouvez',
 'le',
 'voir',
 'ici',
 ',',
 'nous',
 'avons',
 'ah',
 'dans',
 'le',
 'premier',
 'exemple',
 ',',
 'nous',
 'avons',
 'le',
 'terme',
 'emprunté',
 '``',
 'batch',
 'cooking',
 "''",
 'qui',
 'est',
 'un',
 'emprunt',
 'de',
 'plusieurs',
 'mots',
 '.']

In [7]:
src = '- Compared to highly-optimized tiny CNN object detectors, YOLOS achieves competitive performance in terms of AP, FLOPs and FPS. It could serve as a promising starting point for Transformer-based model scaling in object detection. Handling of Variable Input Sizes: \n - Unlike image classification, object detection benchmarks usually have variable image resolutions and aspect ratios.'
tgt = """- 高度に最適化された小型CNNオブジェクト検出器と比較して、YOLOSはAP、FLOPs、FPSの観点で競争力のあるパフォーマンスを達成しています。これは、オブジェクト検出におけるトランスフォーマーベースのモデルスケーリングの有望な出発点となる可能性があります。可変入力サイズの取り扱い：\n - 画像分類とは異なり、オブジェクト検出のベンチマークは通常、可変の画像解像度とアスペクト比を持っています。"""

In [11]:
# pre-processing
# import jieba
# sent_src, sent_tgt = src.strip().split(), [i for i in jieba.cut(tgt, cut_all=False)]

import MeCab
mecab = MeCab.Tagger('-Owakati')
sent_src, sent_tgt = word_tokenize(src.strip()), mecab.parse(tgt.strip()).split()

# sent_src, sent_tgt = src.strip().split(), tgt.strip().split()
token_src, token_tgt = [tokenizer.tokenize(word) for word in sent_src], [tokenizer.tokenize(word) for word in sent_tgt]
wid_src, wid_tgt = [tokenizer.convert_tokens_to_ids(x) for x in token_src], [tokenizer.convert_tokens_to_ids(x) for x in token_tgt]
ids_src, ids_tgt = tokenizer.prepare_for_model(list(itertools.chain(*wid_src)), return_tensors='pt', model_max_length=tokenizer.model_max_length, truncation=True)['input_ids'], tokenizer.prepare_for_model(list(itertools.chain(*wid_tgt)), return_tensors='pt', truncation=True, model_max_length=tokenizer.model_max_length)['input_ids']
sub2word_map_src = []
for i, word_list in enumerate(token_src):
  sub2word_map_src += [i for x in word_list]
sub2word_map_tgt = []
for i, word_list in enumerate(token_tgt):
  sub2word_map_tgt += [i for x in word_list]

# alignment
align_layer = 8
threshold = 1e-4
model.eval()
with torch.no_grad():
  out_src = model(ids_src.unsqueeze(0).to(device), output_hidden_states=True)[2][align_layer][0, 1:-1]
  out_tgt = model(ids_tgt.unsqueeze(0).to(device), output_hidden_states=True)[2][align_layer][0, 1:-1]

  dot_prod = torch.matmul(out_src, out_tgt.transpose(-1, -2))

  softmax_srctgt = torch.nn.Softmax(dim=-1)(dot_prod)
  softmax_tgtsrc = torch.nn.Softmax(dim=-2)(dot_prod)

  softmax_inter = (softmax_srctgt > threshold)*(softmax_tgtsrc > threshold)

align_subwords = torch.nonzero(softmax_inter, as_tuple=False)
align_words = set()
for i, j in align_subwords:
  align_words.add( (sub2word_map_src[i], sub2word_map_tgt[j]) )

# printing
class color:
   PURPLE = '\033[95m'
   CYAN = '\033[96m'
   DARKCYAN = '\033[36m'
   BLUE = '\033[94m'
   GREEN = '\033[92m'
   YELLOW = '\033[93m'
   RED = '\033[91m'
   BOLD = '\033[1m'
   UNDERLINE = '\033[4m'
   END = '\033[0m'

for i, j in sorted(align_words):
  print(f'{color.BOLD}{color.BLUE}{sent_src[i]}{color.END}==={color.BOLD}{color.RED}{sent_tgt[j]}{color.END}')

-===-
Compared===と
Compared===比較
to===と
highly-optimized===高度
highly-optimized===に
highly-optimized===最適
highly-optimized===化
highly-optimized===た
tiny===小型
CNN===CNN
object===オブジェクト
detectors===検出
detectors===器
,===、
YOLOS===YOLOS
achieves===は
achieves===達成
competitive===競争
performance===力
in===の
terms===観点
of===で
AP===AP
,===、
FLOPs===FLOPs
and===、
FPS===FPS
.===。
It===これ
promising===有望
promising===な
starting===出発
point===点
Transformer-based===トランスフォーマー
Transformer-based===ベース
Transformer-based===の
model===モデル
scaling===スケーリング
in===に
in===の
object===オブジェクト
detection===検出
.===。
Handling===取り扱い
of===の
Variable===可変
Input===入力
Sizes===サイズ
:===：
-===-
Unlike===異なり
image===画像
classification===分類
,===、
object===オブジェクト
detection===検出
usually===通常
have===は
variable===可変
image===画像
resolutions===度
and===と
aspect===アスペクト
ratios===比
.===。


In [29]:
len(sent_src), len(sent_tgt)

(54, 101)

In [25]:
align_words

{(0, 0),
 (1, 13),
 (1, 14),
 (2, 13),
 (3, 1),
 (3, 3),
 (3, 4),
 (3, 7),
 (4, 8),
 (5, 9),
 (6, 10),
 (7, 11),
 (7, 12),
 (7, 17),
 (8, 18),
 (9, 19),
 (9, 34),
 (10, 28),
 (11, 29),
 (12, 25),
 (13, 26),
 (14, 27),
 (15, 20),
 (15, 21),
 (16, 22),
 (17, 23),
 (18, 24),
 (18, 39),
 (19, 40),
 (24, 54),
 (24, 55),
 (25, 56),
 (26, 57),
 (28, 48),
 (28, 49),
 (29, 51),
 (30, 52),
 (31, 50),
 (32, 43),
 (33, 44),
 (33, 65),
 (34, 70),
 (35, 69),
 (36, 66),
 (37, 67),
 (38, 68),
 (38, 71),
 (39, 72),
 (40, 77),
 (41, 73),
 (42, 74),
 (42, 78),
 (43, 79),
 (44, 80),
 (46, 85),
 (47, 84),
 (48, 87),
 (49, 89),
 (50, 91),
 (51, 92),
 (52, 93),
 (53, 94),
 (53, 100)}

In [20]:
import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize

mecab = MeCab.Tagger('-Owakati')
            
def get_word_indices(original_sentence):
    # words = original_sentence.strip().split()  # Split words
    words = word_tokenize(original_sentence.strip())
    # words = mecab.parse(original_sentence.strip()).split()
    indices = []
    position = 0  # Track the character position in the original string
    
    for word in words:
        # Locate the word by iterating forward
        while position < len(original_sentence):
            # Check if the substring matches the word
            if original_sentence[position:position+len(word)] == word:
                indices.append([position, position+len(word)])
                position += len(word)  # Move position past the word
                break
            position += 1  # Move forward to next character
    
    return words, indices

sentence = "- Compared to highly-optimized tiny CNN object detectors, YOLOS achieves competitive performance in terms of AP, FLOPs and FPS. It could serve as a promising starting point for Transformer-based model scaling in object detection. Handling of Variable Input Sizes: \n - Unlike image classification, object detection benchmarks usually have variable image resolutions and aspect ratios."
words, indices = get_word_indices(sentence)

print("Words:", words)
print("Indices in original sentence:", indices)


Words: ['-', 'Compared', 'to', 'highly-optimized', 'tiny', 'CNN', 'object', 'detectors', ',', 'YOLOS', 'achieves', 'competitive', 'performance', 'in', 'terms', 'of', 'AP', ',', 'FLOPs', 'and', 'FPS', '.', 'It', 'could', 'serve', 'as', 'a', 'promising', 'starting', 'point', 'for', 'Transformer-based', 'model', 'scaling', 'in', 'object', 'detection', '.', 'Handling', 'of', 'Variable', 'Input', 'Sizes', ':', '-', 'Unlike', 'image', 'classification', ',', 'object', 'detection', 'benchmarks', 'usually', 'have', 'variable', 'image', 'resolutions', 'and', 'aspect', 'ratios', '.']
Indices in original sentence: [[0, 1], [2, 10], [11, 13], [14, 30], [31, 35], [36, 39], [40, 46], [47, 56], [56, 57], [58, 63], [64, 72], [73, 84], [85, 96], [97, 99], [100, 105], [106, 108], [109, 111], [111, 112], [113, 118], [119, 122], [123, 126], [126, 127], [128, 130], [131, 136], [137, 142], [143, 145], [146, 147], [148, 157], [158, 166], [167, 172], [173, 176], [177, 194], [195, 200], [201, 208], [209, 211], 

[nltk_data] Downloading package punkt to /home/jiaruil5/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [17]:
indices_tgt = []
term = "高度に最適化された"
for (i_src, i_tgt) in align_words:
    if words[i_src].lower() in term:
        indices_tgt.append(i_tgt)
indices_tgt

[4, 1, 7, 77, 3, 8, 9, 14, 13, 12, 10, 2, 13, 11]

In [21]:
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')

def custom_tokenize(text, preserve_list):
    # Create placeholders for terms to preserve
    placeholders = {term: f"__PLACEHOLDER_{i}__" for i, term in enumerate(preserve_list)}
    
    # Replace terms in preserve_list with placeholders
    for term, placeholder in placeholders.items():
        text = text.replace(term, placeholder)
    
    # Tokenize the text
    tokens = word_tokenize(text)
    
    # Replace placeholders with original terms
    tokens = [term if term in placeholders.values() else token for token in tokens]
    for placeholder, term in placeholders.items():
        tokens = [token.replace(term, placeholder) if term in token else token for token in tokens]
    
    return tokens

# Example usage
text = "This is a sample sentence containing special terms like New York and machine learning."
preserve_list = ["New York", "machine learning"]

tokens = custom_tokenize(text, preserve_list)
print(tokens)


['This', 'is', 'a', 'sample', 'sentence', 'containing', 'special', 'terms', 'like', 'New York', 'and', 'machine learning', '.']


[nltk_data] Downloading package punkt to /home/jiaruil5/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
